# Simulations of Park, Wong-Campos et al. (2025): the published NEURON model

Paper: [doi:10.1038/s41467-025-55819-9](https://doi.org/10.1038/s41467-025-55819-9). The model
claims use the paper's Supplementary Software 1 (41467_2025_55819_MOESM7_ESM.zip, sha256
9eab52ac...7049249, as in the claims file), downloaded from the journal and run here. It holds no
data from the dandiset (DANDI:001266, version 0.250710.0758, used by the other notebook of this
entry).

Changes needed to run it, none of which touches the model:
- NEURON 9 (the only version with wheels for this Python) refuses non-ASCII characters in mod
  files; they occur only in comments and are removed. The paper used NEURON 8.2.
- `run_morphology.py` calls `FullCA1Model(channel_params, morphology_file)`, but the class needs
  a third argument, `ka_cutoff`; the 100 µm that `run_simulation.py` passes is supplied.
- The scripts' plotting is not run; their simulation calls are, with the parameters of
  `main.py` and the two `parameters_list.py` files.

| Claim | Tested here |
|---|---|
| fig4f-g-model-accelerometer | tested |
| figs20ab-model-patch-vs-opto | tested |
| figs20-model-robustness | blocked, too large (estimate below) |
| figs21-izhikevich | blocked, MATLAB script (see the attempt record) |

In [ ]:
import hashlib
import importlib.util
import io
import os
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile

import matplotlib
import numpy as np
import requests

URL = ("https://static-content.springer.com/esm/art%3A10.1038%2Fs41467-025-55819-9/MediaObjects/"
       "41467_2025_55819_MOESM7_ESM.zip")
SHA256 = "9eab52ac20d889991fc7ba5f24130d0d80f7d53c3ea69057c5af0f4007049249"
blob = requests.get(URL, timeout=120).content
print("sha256 matches the claims file's prefix and suffix:", hashlib.sha256(blob).hexdigest() == SHA256)
work = tempfile.mkdtemp()
root = "Supplementary NEURON models/Conductance-based model/"
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    for name in z.namelist():
        if not name.startswith(root) or name.endswith("/"):
            continue
        rel = name[len(root):]
        top = rel.split("/")[0]
        if rel.endswith(".py") and "/" not in rel:
            target = os.path.join(work, rel)
        elif rel.startswith("Model Robustness/Channels") and rel.endswith((".mod", ".inc")):
            target = os.path.join(work, os.path.basename(rel))
        elif rel.endswith("parameters_list.py"):
            target = os.path.join(work, "params_" + ("robustness" if top == "Model Robustness" else "oblique") + ".py")
        else:
            continue
        data = z.read(name)
        if target.endswith((".mod", ".inc")):
            data = bytes(c for c in data if c < 128)  # NEURON 9 rejects non-ASCII (comments only)
        with open(target, "wb") as f:
            f.write(data)
nrnivmodl = os.path.join(os.path.dirname(sys.executable), "nrnivmodl")
build = subprocess.run([nrnivmodl], cwd=work, capture_output=True, text=True)
print("nrnivmodl exit code:", build.returncode)
os.chdir(work)
sys.path.insert(0, work)
matplotlib.use("Agg")

import neuron  # noqa: E402  (after the mechanisms are built in the working directory)
import run_morphology as rm  # noqa: E402
from FullCA1Model import FullCA1Model  # noqa: E402
from delete_neuron import delete_all_sections  # noqa: E402
from run_simulation import reshape_v, run_simulation  # noqa: E402

rm.FullCA1Model = lambda cp, mf: FullCA1Model(cp, mf, 100)  # ka_cutoff as in run_simulation.py

def load_params(name):
    spec = importlib.util.spec_from_file_location(name, os.path.join(work, name + ".py"))
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.setup_parameters

# Channel parameters of main.py
NAV_INACTIVATION = [1, 0.6, 300, 100]
KDR = [0.04, 0.04, 0, 1]
OFF = lambda d, z: [1e-9, 1e-9, d, z]

def simulate(setup_parameters, run_nr):
    (duration, stim_start, stim_end, chr_params, synapse_params, sim_params, outputs, waveform, waveform_patch,
     patch_params, nav, kad, kap, cal, car) = setup_parameters(run_nr)
    channel_params = [nav, NAV_INACTIVATION, kap, kad, KDR, chr_params, OFF(550, 100), OFF(300, 100), cal,
                      OFF(50, 30), OFF(300, 100), OFF(550, 100), car, OFF(550, 100), OFF(200, 100),
                      OFF(200, 100), OFF(550, 100)]
    morpho = rm.run_morphology(channel_params, "Jarsky", "segments")
    delete_all_sections()
    dist = [[float(x) for x in row] for row in morpho[1]]
    result = run_simulation(sim_params, channel_params, synapse_params, waveform, ["v"], "Jarsky", patch_params,
                            dist, "segments")
    delete_all_sections()
    t = np.array([float(x) for x in result[0]])
    flat = []
    for item in result[1]["v"]:
        flat.extend(float(x) for x in item)
    v = np.array(reshape_v(flat, dist))
    return dict(t=t, v=v, D=np.array(dist[0]), path=np.array(morpho[3]), names=list(morpho[0]),
                nav=nav, kad=kad, stim=(stim_start, stim_end))

robust = load_params("params_robustness")
oblique = load_params("params_oblique")
t0 = time.time()
runs = {"soma step": simulate(robust, 3), "wide-field step": simulate(robust, 12),
        "patch, oblique": simulate(oblique, 0), "optogenetic, oblique": simulate(oblique, 1)}
print(f"four runs in {time.time() - t0:.0f} s")

In [ ]:
print("NEURON", neuron.__version__)
for name, r in runs.items():
    print(f"{name}: {r['v'].shape[0]} compartments, {len(r['t'])} time points to {r['t'][-1]:.0f} ms, "
          f"distance 0 to {r['D'].max():.0f} µm, main path {len(r['path'])} compartments, "
          f"v {r['v'].min():.1f} to {r['v'].max():.1f} mV, NaV {r['nav']}, KA {r['kad']}")

Membrane potentials stay between about -80 and +50 mV and the morphology spans 0 to about 840 µm
from the soma, as for a CA1 pyramidal cell. The "Model Robustness" parameters give NaV 50 and 48
mS/cm² (soma and distal); Table 1 gives 50 and 47.

## fig4f-g-model-accelerometer

> In the multi-compartment CA1 model, step stimulation at the soma gives a dSpike motif of
> failure, success, success, failures, and wide-field stimulation gives alternating successes
> and failures followed by repeated failures.

Runs "Optopatch step soma" and "Optopatch step widefield" of `parameters_list.py` (Model
Robustness folder, the only one with these runs). A bAP succeeds when the compartment of the main
apical path nearest 500 µm from the soma peaks above -40 mV before the next somatic spike
(Methods, Parameter fitting). Somatic spikes are upward crossings of 0 mV.

In [ ]:
def motif(r):
    t, v, D, path = r["t"], r["v"], r["D"], r["path"]
    soma = int(np.argmin(D))
    j = int(path[np.argmin(np.abs(D[path] - 500))])
    up = np.flatnonzero((v[soma, 1:] >= 0) & (v[soma, :-1] < 0)) + 1
    seq, peaks = "", []
    for k, u in enumerate(up):
        end = up[k + 1] if k + 1 < len(up) else len(t)
        pk = v[j, u:end].max()
        peaks.append(round(float(pk), 1))
        seq += "S" if pk > -40 else "F"
    return dict(compartment_um=round(float(D[j]), 1), spikes_ms=np.round(t[up], 1).tolist(), peaks_mV=peaks, motif=seq)

for name in ["soma step", "wide-field step"]:
    m = motif(runs[name])
    print(name, "| stimulus", runs[name]["stim"], "ms |", m)

**Result: not reproduced.** The soma step gives success, success, success, success, failure (5
somatic spikes), not failure, success, success, failures: the first bAP already reaches above
-40 mV at 494 µm. The wide-field step gives three successes, then alternating failures and
successes to the end of the 90 ms run (13 spikes), so the alternation appears but the opening
failure and the closing repeated failures do not.

## figs20ab-model-patch-vs-opto

> In the model, a 2 nA dendritic current injection evokes dendrite-localized sodium spikes,
> while optogenetic stimulation of the same dendrite evokes a somatic spike first that then
> back-propagates.

Runs "Patch pulse oblique" and "Optopatch pulse oblique" of the "Oblique Patch Clamp and
Optogenetic Stimulus" folder (section 98 in both). For each run, the compartment that first
crosses 0 mV ("peaks first" read as "first reaches 0 mV"), the first outside the axon, and the
times at which the soma and the stimulated dendrite (where the current injection first spikes)
first do.

In [ ]:
AXON = ("axon", "node", "inode", "hill", "ais", "myelin")
site = None
for name in ["patch, oblique", "optogenetic, oblique"]:
    r = runs[name]
    t, v, D, names = r["t"], r["v"], r["D"], r["names"]
    cross = np.array([np.argmax(row >= 0) if (row >= 0).any() else len(t) for row in v])
    tc = lambda i: float(t[cross[i]]) if cross[i] < len(t) else np.nan
    dend = np.array([not n.lower().startswith(AXON) for n in names])
    first_any = int(np.argmin(cross))
    first_dend = int(np.flatnonzero(dend)[np.argmin(cross[dend])])
    soma = int(np.argmin(D))
    if site is None:
        site = first_dend  # the stimulated dendrite: where the current injection first spikes
    print(f"{name}: first compartment over 0 mV {names[first_any]} at {tc(first_any):.2f} ms ({D[first_any]:.0f} µm); "
          f"first outside the axon {names[first_dend]} at {tc(first_dend):.2f} ms ({D[first_dend]:.0f} µm); "
          f"soma {names[soma]} at {tc(soma):.2f} ms; stimulated site {names[site]} at {tc(site):.2f} ms; "
          f"compartments over 0 mV: {int((cross < len(t)).sum())} of {len(cross)}")

**Result: reproduced.** The current injection into the oblique dendrite drives that dendrite
(apic[23], 186 µm from the soma) over 0 mV at 15.02 ms while the soma never reaches 0 mV: a
dendrite-localized spike (10 of 1265 compartments over 0 mV). The optogenetic stimulus of the same
dendrite first drives the axon (a node, 25.02 ms) and the initial segment (25.10 ms), then the
soma (25.35 ms) over 0 mV, and 658 compartments follow, while the stimulated dendrite itself does
not reach 0 mV: the spike starts perisomatically, in the axon a fraction of a millisecond before
the soma, and back-propagates.

## figs20-model-robustness (not run)

The phase diagrams of Fig. S20c-e are parameter sweeps; the plotting script lists 11 × 14 = 154
runs for Fig. S20c alone. One 100 ms run of the same kind is timed here to estimate the cost.

In [ ]:
t0 = time.time()
_ = simulate(robust, 3)
one = time.time() - t0
print(f"one run: {one:.1f} s; 154 runs for Fig. S20c: about {154 * one / 60:.0f} min; the three diagrams together several times that")

## Summary

| Claim | Outcome |
|---|---|
| fig4f-g-model-accelerometer | not reproduced |
| figs20ab-model-patch-vs-opto | reproduced |
| figs20-model-robustness | blocked, too large |
| figs21-izhikevich | blocked, not runnable (MATLAB) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.